## Imports og tilkobling

In [1]:
import sys
import os
import pandas as pd
import io
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

sys.path.append(os.path.abspath("../../../"))
from src.feature_engineering.forbruksdata import create_consumption_features

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_name = "rawdata"

## Hente CSV fra blob

In [2]:
blob_name = "AntallMålepunktNorgespris.csv"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=blob_name
)

data = blob_client.download_blob().readall()

df = pd.read_csv(io.BytesIO(data))

## Feature engineering

In [3]:
from src.feature_engineering.norgespris import clean_norgespris_df, split_by_station

df = clean_norgespris_df(df)

from src.feature_engineering.norgespris import (
    clean_norgespris_df,
    split_by_station,
    fill_missing_days_per_station
)

df = clean_norgespris_df(df)
station_dfs = split_by_station(df)

periods = [
    ("2024-11-01", "2025-01-31"),
    ("2025-11-01", "2026-01-31")
]

station_dfs = fill_missing_days_per_station(station_dfs, periods)

df.head()

,transformer_station,timestamp,count_total
0,TIMENES TS,2025-10-01,1946
1,TIMENES TS,2025-10-02,1990
2,TIMENES TS,2025-10-03,2022
3,TIMENES TS,2025-10-04,2040
4,TIMENES TS,2025-10-05,2055


## Lagre i Azure Blob

In [4]:
output_container = "processeddata"

for station_name, station_df in station_dfs.items():
    
    output_blob_name = f"{station_name}_norgespris.csv"
    
    csv_buffer = io.StringIO()
    station_df.to_csv(csv_buffer, index=False)

    blob_client = blob_service_client.get_blob_client(
        container=output_container,
        blob=output_blob_name
    )

    blob_client.upload_blob(csv_buffer.getvalue(), overwrite=True)

    print(f"Lastet opp: {output_blob_name}")

Lastet opp: breive_norgespris.csv
Lastet opp: frikstad_norgespris.csv
Lastet opp: hartevatn_norgespris.csv
Lastet opp: timenes_ts_norgespris.csv


# Kvalitetssikring

## Kun en stasjon per fil

In [5]:
for name, df_station in station_dfs.items():
    unique_stations = df_station["transformer_station"].unique()
    
    print(f"\n{name}")
    print("Antall stasjoner:", len(unique_stations))
    print("Stasjon:", unique_stations)


breive
Antall stasjoner: 1
Stasjon: <StringArray>
['BREIVE']
Length: 1, dtype: str

frikstad
Antall stasjoner: 1
Stasjon: <StringArray>
['FRIKSTAD']
Length: 1, dtype: str

hartevatn
Antall stasjoner: 1
Stasjon: <StringArray>
['HARTEVATN']
Length: 1, dtype: str

timenes_ts
Antall stasjoner: 1
Stasjon: <StringArray>
['TIMENES TS']
Length: 1, dtype: str


## Sjekk timestamp format

In [6]:
for name, df_station in station_dfs.items():
    print(f"\n{name}")
    print(df_station["timestamp"].dtype)
    print(df_station["timestamp"].head(3))


breive
datetime64[ns]
0   2025-11-01
1   2025-11-02
2   2025-11-03
Name: timestamp, dtype: datetime64[ns]

frikstad
datetime64[ns]
0   2025-11-01
1   2025-11-02
2   2025-11-03
Name: timestamp, dtype: datetime64[ns]

hartevatn
datetime64[ns]
0   2025-11-01
1   2025-11-02
2   2025-11-03
Name: timestamp, dtype: datetime64[ns]

timenes_ts
datetime64[ns]
0   2025-11-01
1   2025-11-02
2   2025-11-03
Name: timestamp, dtype: datetime64[ns]


## Sjekk duplikater

In [7]:
for name, df_station in station_dfs.items():
    duplicates = df_station.duplicated(subset=["timestamp", "count_total"]).sum()
    
    print(f"\n{name}")
    print("Duplikater:", duplicates)


breive
Duplikater: 0

frikstad
Duplikater: 0

hartevatn
Duplikater: 0

timenes_ts
Duplikater: 0


## Sjekk manglende verdier

In [8]:
for name, df_station in station_dfs.items():
    print(f"\n{name}")
    print(df_station.isna().sum())


breive
timestamp              0
transformer_station    0
count_total            0
dtype: int64

frikstad
timestamp              0
transformer_station    0
count_total            0
dtype: int64

hartevatn
timestamp              0
transformer_station    0
count_total            0
dtype: int64

timenes_ts
timestamp              0
transformer_station    0
count_total            0
dtype: int64


## Sjekk nullverdier

In [10]:
for name, df_station in station_dfs.items():
    zeros = (df_station["count_total"] == 0).sum()
    
    print(f"\n{name}")
    print("Nullverdier:", zeros)


breive
Nullverdier: 0

frikstad
Nullverdier: 0

hartevatn
Nullverdier: 0

timenes_ts
Nullverdier: 0


## Sjekk hull i tidsserie

In [9]:
import pandas as pd

# Definer perioden
start_date = '2025-11-01'
end_date = '2026-01-31'
full_days = pd.date_range(start=start_date, end=end_date, freq='D')

for name, df_station in station_dfs.items():
    # Normaliser timestamp til dato (00:00:00)
    df_station['date'] = pd.to_datetime(df_station['timestamp']).dt.normalize()
    
    # Hent unike dager
    station_days = df_station['date'].drop_duplicates()
    
    # Sjekk hvilke dager som mangler
    missing_days = full_days.difference(station_days)
    
    if len(missing_days) == 0:
        print(f"{name}: Ingen dager mangler ✅")
    else:
        print(f"{name}: Mangler {len(missing_days)} dag(er): {missing_days.date.tolist()}")

breive: Ingen dager mangler ✅
frikstad: Ingen dager mangler ✅
hartevatn: Ingen dager mangler ✅
timenes_ts: Ingen dager mangler ✅


## Sjekk fordeling

In [11]:
for name, df_station in station_dfs.items():
    print(f"\n{name}")
    print(df_station["count_total"].describe())


breive
count      92.000000
mean     1102.184783
std        31.830226
min      1035.000000
25%      1078.750000
50%      1109.000000
75%      1132.250000
max      1145.000000
Name: count_total, dtype: float64

frikstad
count      92.000000
mean     4893.532609
std       174.609483
min      4521.000000
25%      4777.000000
50%      4929.000000
75%      5033.250000
max      5138.000000
Name: count_total, dtype: float64

hartevatn
count      92.000000
mean     1862.184783
std        39.318907
min      1769.000000
25%      1839.750000
50%      1866.000000
75%      1890.250000
max      1919.000000
Name: count_total, dtype: float64

timenes_ts
count      92.000000
mean     2629.010870
std       110.523777
min      2391.000000
25%      2550.000000
50%      2648.500000
75%      2720.500000
max      2785.000000
Name: count_total, dtype: float64


## Sjekk endring over tid

In [12]:
for name, df_station in station_dfs.items():
    df_station = df_station.sort_values("timestamp")
    df_station["diff"] = df_station["count_total"].diff()

    print(f"\n{name}")
    print(df_station["diff"].describe())


breive
count    91.000000
mean      1.208791
std       1.303981
min       0.000000
25%       0.000000
50%       1.000000
75%       2.000000
max       7.000000
Name: diff, dtype: float64

frikstad
count    91.00000
mean      6.78022
std       4.79769
min       0.00000
25%       3.00000
50%       5.00000
75%      10.50000
max      19.00000
Name: diff, dtype: float64

hartevatn
count    91.000000
mean      1.648352
std       1.702114
min       0.000000
25%       0.000000
50%       1.000000
75%       3.000000
max       7.000000
Name: diff, dtype: float64

timenes_ts
count    91.00000
mean      4.32967
std       3.13069
min       0.00000
25%       2.00000
50%       3.00000
75%       7.00000
max      13.00000
Name: diff, dtype: float64
